# An Attribution-Controlled Study of Feature Priors in LLM-Guided Grammatical Evolution

Driver notebook. All logic lives in the `geprior` package; this notebook orchestrates the study and reports results. Every stage writes to a checkpoint and skips work already done, so the notebook can be interrupted and re-run without recomputing completed cells.

The campaign runs at `config.N_SEEDS` seeds (30 by default). For a quick end-to-end check in VS Code, set `SMOKE = True` in the next cell: it shrinks the search and the seed count and writes to a separate scratch folder, leaving the full-run checkpoints untouched.

In [ ]:
import sys
from pathlib import Path

# --- locate the package, wherever the kernel's working directory happens to be ---
_SKIP = {".git", ".ipynb_checkpoints", "__pycache__", "node_modules", ".venv",
         "venv", "env", ".idea", ".vscode", "data", "results", "figures"}


def _search_downwards(base, package, max_depth=3):
    frontier, depth = [Path(base)], 0
    while frontier and depth <= max_depth:
        nxt = []
        for folder in frontier:
            if (folder / package / "__init__.py").exists():
                return folder
            try:
                children = [c for c in folder.iterdir()
                            if c.is_dir() and c.name not in _SKIP
                            and not c.name.startswith(".")]
            except (PermissionError, OSError):
                continue
            nxt.extend(children)
        frontier, depth = nxt, depth + 1
    return None


def add_repository_root(package="geprior"):
    """Put the repository root on sys.path. Jupyter and VS Code disagree about a
    notebook's working directory, so both directions are searched."""
    roots = [Path.cwd().resolve()]
    notebook = globals().get("__vsc_ipynb_file__")     # set by VS Code only
    if notebook:
        roots.append(Path(notebook).resolve().parent)
    for root in roots:
        for base in [root, *root.parents]:
            if (base / package / "__init__.py").exists():
                if str(base) not in sys.path:
                    sys.path.insert(0, str(base))
                return base
    for root in roots:
        found = _search_downwards(root, package)
        if found is not None:
            if str(found) not in sys.path:
                sys.path.insert(0, str(found))
            return found
    raise ModuleNotFoundError(
        f"could not locate the {package!r} package. Unpack the full project so that "
        f"the {package}/ folder sits alongside this notebook, or add its parent with "
        f"sys.path.insert(0, r'C:\\path\\to\\ge_prior_attribution').")


ROOT = add_repository_root()

# --- install any missing dependency into the interpreter actually running here ---
REQUIRED_PACKAGES = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
                     "sklearn": "scikit-learn", "deap": "deap",
                     "matplotlib": "matplotlib"}


def ensure_dependencies():
    import importlib, importlib.util, subprocess
    missing = [pip_name for module, pip_name in REQUIRED_PACKAGES.items()
               if importlib.util.find_spec(module) is None]
    if not missing:
        return
    print("kernel interpreter:", sys.executable)
    print("installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    importlib.invalidate_caches()


ensure_dependencies()

import numpy as np
import pandas as pd
from geprior import config, datasets, priors, grammar, engine
from geprior import experiments, baselines, statistics as st, figures as fg
pd.set_option("display.width", 160)

# --- quick-check toggle -------------------------------------------------------
# SMOKE = True runs a tiny version end to end and writes to results/_smoke so it
# does not touch the full-campaign checkpoints. Set to False for the real 30-seed run.
SMOKE = False
if SMOKE:
    import dataclasses
    config.GE = dataclasses.replace(config.GE, population_size=40, n_generations=6,
                                    hall_of_fame_size=8)
    config.N_SEEDS = 3
    config.ABLATION_SEEDS = 2
    config.ENSEMBLE_SIZE = 5
    config.RESULTS_DIR = config.RESULTS_DIR / "_smoke"
    config.FIGURES_DIR = config.FIGURES_DIR / "_smoke"
    config.RESULTS_DIR.mkdir(exist_ok=True)
    config.FIGURES_DIR.mkdir(exist_ok=True)

KEYS = ["wbcd", "pima", "cleveland", "heart_failure"]
ARMS = list(config.SELECTION_ARMS)
CONSTRUCTION = list(config.CONSTRUCTION_ARMS)
SEEDS = range(1, config.N_SEEDS + 1)
print("repository root:", ROOT)
print(config.summary())

## 1. Datasets

Four public clinical datasets spanning a range of difficulty, so an attribution effect that only appears away from a performance ceiling is not masked by a saturated benchmark. `K` is the fixed selection budget shared by every selection-prior arm.

In [ ]:
datasets.describe()

## 2. The knowledge prior

The prior is an LLM output reasoning over feature names alone, with no access to labels or data statistics. `LLM_BACKEND = "cached"` replays a provenance-documented response; setting it to `"api"` with `ANTHROPIC_API_KEY` draws a fresh response per seed.

In [ ]:
print("backend:", priors.LLM.backend, "| live:", priors.LLM.is_live)
print("provenance:", priors.CACHED_LLM_PRIOR["provenance"])
for key in KEYS:
    split = datasets.prepare_split(key, seed=1)
    idx = priors.selection_indices("llm", split.X_train, split.y_train,
                                   split.features, split.budget, 1, key)
    print(f"  {key:14s}", [split.features[i] for i in idx])

## 3. Grammar

A prior is injected purely by restricting the terminal rule `<v>`; every other production is identical across arms, so the only degree of freedom that differs between conditions is which feature indices the grammar can reach.

In [ ]:
split = datasets.prepare_split("pima", seed=1)
idx = priors.selection_indices("llm", split.X_train, split.y_train,
                               split.features, split.budget, 1, "pima")
print(grammar.weighted_grammar_text(idx))

## 4. Baselines

Five standard classifiers on the full unrestricted feature set, fitted on the identical per-seed splits as every GE condition, so the tables are directly comparable rather than computed on a different resampling.

In [ ]:
base = baselines.run_baselines(seeds=SEEDS)
base.groupby(["model", "dataset"])["roc_auc"].median().unstack("dataset")[KEYS].round(3)

### Black-box feature-engineering comparator

A lightweight, faithful-in-spirit reimplementation of the black-box LLM feature-engineering paradigm (CAAFE; Hollmann et al., 2023): LLM-proposed features added greedily under a cross-validated acceptance gate, feeding a logistic regression.

In [ ]:
blackbox = baselines.run_blackbox_baseline(seeds=SEEDS)
blackbox.groupby("dataset")["roc_auc"].median().reindex(KEYS).round(3)

## 5. Selection study

Noise, data and knowledge as sources of a feature prior, at matched budget. Pass `time_budget=<seconds>` to run in bounded slices; re-run the cell to continue from the checkpoint.

In [ ]:
selection = experiments.run_selection_study(seeds=SEEDS, n_restarts=1)
st.median_table(selection, ARMS, dataset_keys=KEYS).round(3)

## 6. Construction study

GE runs unrestricted over the raw feature set augmented with five engineered features. Constructions are computed on raw-scale values and the augmented matrix is re-standardised on training rows only.

In [ ]:
construction = experiments.run_construction_study(seeds=SEEDS)
st.median_table(construction, CONSTRUCTION, dataset_keys=KEYS).round(3)

## 7. Selection-prior figure

In [ ]:
fg.plot_arm_medians(selection, ARMS, KEYS, "selection_by_dataset",
                    title="Selection-prior arms by dataset (median over seeds)").round(3)

## 8. Statistical protocol

Per-dataset Friedman tests over the seed-paired resamples carry the inference. With only four datasets a cross-dataset critical-difference diagram would be under-powered, so the mean rank is reported descriptively and no CD diagram is produced.

In [ ]:
friedman, ranks = st.per_dataset_friedman(selection, ARMS, dataset_keys=KEYS)
display(friedman.round(4))
print("mean rank across datasets (descriptive only):")
print(ranks.mean(axis=0).round(2).to_string())

In [ ]:
fg.plot_rank_heatmap(ranks.reindex(KEYS))

### Pre-registered contrasts

C1 knowledge versus noise, C2 knowledge versus data, C3 the construction analogue of C1. Fixed before any run, Holm-corrected within each dataset, Cliff's delta reported with every pairwise claim.

In [ ]:
contrasts = st.preregistered_contrasts(selection, construction, dataset_keys=KEYS)
contrasts.round(4)

## 9. Structural diagnostics

Diagnostics pooled across datasets: effective length is used codons, structural diversity is the proportion of distinct derivation structures in the final population.

In [ ]:
display(st.structural_diagnostics(selection, ARMS))
fg.plot_length_vs_generalisation(selection, ARMS)

## 10. What interpretability costs

A single symbolic rule against a bagged black-box comparator conflates two things: the cost of the symbolic representation and the cost of comparing one model to an ensemble. Reporting a protocol-matched GE ensemble alongside the single rule separates them.

In [ ]:
ensemble = experiments.run_selection_study(
    seeds=SEEDS, arms=["llm"], n_restarts=5,
    checkpoint=config.RESULTS_DIR / "ensemble.csv")
comparison = pd.DataFrame({
    "GE rule (llm)": selection[selection.arm == "llm"].groupby("dataset")["test_roc_auc"].median(),
    "GE ensemble (llm)": ensemble.groupby("dataset")["ensemble_roc_auc"].median(),
    "Black-box FE": blackbox.groupby("dataset")["roc_auc"].median(),
    "Logistic Regression": base[base.model == "Logistic Regression"].groupby("dataset")["roc_auc"].median(),
}).reindex(KEYS)
fg.plot_interpretability_cost(comparison)
comparison.round(3)

## 11. Configuration ablation

Each variant adds exactly one configuration choice to the one above it, so the effect is attributed to a specific choice rather than to the pipeline as a whole.

In [ ]:
ablation = experiments.run_ablation(seeds=range(1, config.ABLATION_SEEDS + 1))
display(ablation.groupby(["dataset", "variant"])["test_roc_auc"].median()
        .unstack("variant").reindex(KEYS).round(3))
display(ablation.groupby(["dataset", "variant"])["test_f1"].median()
        .unstack("variant").reindex(KEYS).round(3))
fg.plot_configuration_ablation(ablation)

## 12. Representative interpretable rules

The run whose test ROC-AUC is nearest that arm's **median**, not its best. Selecting the best-scoring or highest-train-accuracy run would showcase an optimistic outlier.

In [ ]:
def representative_rule(df, key, arm="llm"):
    sub = df[(df.dataset == key) & (df.arm == arm)].dropna(subset=["test_roc_auc"])
    target = sub["test_roc_auc"].median()
    row = sub.iloc[(sub["test_roc_auc"] - target).abs().argmin()]
    split = datasets.prepare_split(key, int(row["seed"]))
    return dict(dataset=key, seed=int(row["seed"]),
                roc_auc=round(row["test_roc_auc"], 3), f1=round(row["test_f1"], 3),
                effective_length=int(row["effective_length"]),
                rule=engine.readable_rule(row["phenotype"], split.features))


for key in KEYS:
    r = representative_rule(selection, key)
    print(f"\n{r['dataset']}  AUC={r['roc_auc']}  F1={r['f1']}  len={r['effective_length']}")
    print("  ", r["rule"][:220])

## 13. Explainability check on WBCD

A surrogate explainer. The SHAP values below are computed on a **random forest**, not on the evolved rule: they are an independent check on which features a conventional black-box model considers important, and must not be described as an explanation of the evolved model itself.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

rep = representative_rule(selection, "wbcd")
split = datasets.prepare_split("wbcd", rep["seed"])
row = selection[(selection.dataset == "wbcd") & (selection.arm == "llm")
                & (selection.seed == rep["seed"])].iloc[0]
scores = engine.phenotype_score(row["phenotype"], split.X_test)
fg.plot_confusion(split.y_test, (scores > row["threshold"]).astype(int),
                  "WBCD evolved rule, test partition", "wbcd_confusion")
forest = RandomForestClassifier(n_estimators=400,
                                random_state=config.GLOBAL_SEED).fit(split.X_train, split.y_train)
order = np.argsort(forest.feature_importances_)[::-1][:6]
print("random-forest surrogate, top-6 features:", [split.features[i] for i in order])
print("features referenced by the evolved rule:",
      [split.features[i] for i in engine.used_feature_indices(row["phenotype"], split.n_features)])

## 14. Consolidated summary

In [ ]:
print("=== baselines ===")
print(base.groupby(["model", "dataset"])["roc_auc"].median().unstack("dataset")[KEYS].round(3).to_string())
print("\n=== selection arms (median test ROC-AUC) ===")
print(st.median_table(selection, ARMS, dataset_keys=KEYS).round(3).to_string())
print("\n=== construction arms (median test ROC-AUC) ===")
print(st.median_table(construction, CONSTRUCTION, dataset_keys=KEYS).round(3).to_string())
print("\n=== per-dataset Friedman ===")
print(friedman.round(4).to_string(index=False))
print("\n=== pre-registered contrasts ===")
print(contrasts.round(4).to_string(index=False))
print("\n=== structural diagnostics ===")
print(st.structural_diagnostics(selection, ARMS).to_string())
print("\n=== interpretability cost ===")
print(comparison.round(3).to_string())